# 3) Machine learning: gradient boosting, features and leaks

## Objectives

At the end of this notebook you should be able to:

- train a **gradient-boosting** forecaster with scikit-learn and compare it fairly with lesson 2's regression
- build **lag features** that respect the day-ahead information set — and test each one for availability
- recognise **leakage through features**: a feature that looks harmless in a table but uses the future
- validate in time: why **shuffled cross-validation** flatters a forecaster, and what to do instead
- combine models in an **ensemble**, and explain why it can beat every member

Machine learning promises to find patterns no one specified. For forecasting it brings two dangers: models flexible enough to exploit any leak, and evaluation habits (random splits, shuffled folds) borrowed from problems where time does not exist. This lesson trains a gradient-boosting model on the same honest information as lesson 2 — and finds out what it is really worth.

In [1]:
import logging
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import KFold, TimeSeriesSplit

from forecast_data import load_data, temperature_forecast

warnings.filterwarnings("ignore")
logging.getLogger("sklearn").setLevel(logging.ERROR)

### Features, and what is known at noon

In [2]:
df = load_data()
df["temperature_forecast"] = temperature_forecast(df)
demand, idx = df["demand"], df.index
dow = idx.dayofweek
daytype = np.where(df.holiday | (dow == 6), 2, np.where(dow == 5, 1, 0))      # 0 workday, 1 Saturday, 2 Sunday/holiday

issue = idx.normalize() - pd.Timedelta(hours=13)                               # 11:00 on D-1: the last known hour
features = pd.DataFrame({
    "hour": idx.hour, "daytype": daytype, "day_of_year": idx.dayofyear,
    "temperature_forecast": df.temperature_forecast,
    "last_known": demand.reindex(issue).to_numpy(),                            # demand at 11:00 on D-1
    "same_hour_2_days_ago": demand.shift(48),
    "same_hour_last_week": demand.shift(168),
}, index=idx)
LAGS = {"last_known": None, "same_hour_2_days_ago": 48, "same_hour_last_week": 168}


def available(lag_hours, target_hour, issue_hour=12):
    """Is the value lag_hours before hour target_hour of day D known at issue_hour on D-1?"""
    hours_ahead = target_hour + 25 - issue_hour           # from the last known hour (issue_hour - 1 on D-1)
    return lag_hours >= hours_ahead


for name, lag in LAGS.items():
    if lag is not None:
        assert all(available(lag, h) for h in range(24)), name
print("every lag feature is known at noon on D-1 for all 24 hours of D")
print("leaky candidates: 1-hour lag available for", sum(available(1, h) for h in range(24)),
      "hours, 24-hour lag for", sum(available(24, h) for h in range(24)), "hours")
features.dropna().head(3)

every lag feature is known at noon on D-1 for all 24 hours of D
leaky candidates: 1-hour lag available for 0 hours, 24-hour lag for 12 hours


                     hour  daytype  day_of_year  temperature_forecast  \
2023-01-08 00:00:00     0        2            8             -0.042531   
2023-01-08 01:00:00     1        2            8              0.689548   
2023-01-08 02:00:00     2        2            8              1.062748   

                      last_known  same_hour_2_days_ago  same_hour_last_week  
2023-01-08 00:00:00  2212.554368           1753.011940          1891.814698  
2023-01-08 01:00:00  2212.554368           1789.583511          1870.222676  
2023-01-08 02:00:00  2212.554368           1815.906628          1867.903308  

Three kinds of features, all known at noon on D−1: the calendar (hour, type of day with holidays as Sundays, day of the year), the **temperature forecast**, and three demand lags. The function `available` turns lesson 1's rule into a test: a value is usable only if its lag is at least the forecast's lead time, 13 to 36 hours. The tempting 1-hour lag is available for **no** hour of a day-ahead forecast, the 24-hour lag only for the morning — which is why lesson 1's honest "yesterday" had to switch to two days ago at noon.

### Gradient boosting, step by step

`HistGradientBoostingRegressor` builds a sequence of small decision trees, each correcting the errors of the ones before. It needs no feature scaling, finds non-linear effects and interactions by itself — the temperature thresholds, the different shape of Sundays — and trains in seconds on 17,000 hours.

In [3]:
data = features.dropna()
y = demand.loc[data.index]
train, test = data.index < "2025", data.index >= "2025"


def boosted(columns):
    return HistGradientBoostingRegressor(max_iter=1500, learning_rate=0.02, random_state=0).fit(data.loc[train, columns], y[train])


steps = {"calendar": ["hour", "daytype", "day_of_year"],
         "+ temperature forecast": ["hour", "daytype", "day_of_year", "temperature_forecast"],
         "+ demand lags (all features)": list(features.columns)}
rows, models = {}, {}
for name, columns in steps.items():
    start = time.perf_counter()
    models[name] = boosted(columns)
    rows[name] = {"MAE_2025": np.abs(models[name].predict(data.loc[test, columns]) - y[test]).mean(),
                  "train_s": time.perf_counter() - start}
gbm = models["+ demand lags (all features)"]
gbm_forecast = pd.Series(gbm.predict(data.loc[test]), index=data.index[test])

reg_data = df.assign(daytype=np.array(["workday", "saturday", "sunday_or_holiday"])[daytype], hour=idx.hour, month=idx.month)
regression = smf.ols("demand ~ C(hour):C(daytype) + C(month) + bs(temperature, df=6, lower_bound=-12, upper_bound=40)",
                     reg_data.loc[:"2024"]).fit()
reg_forecast = regression.predict(reg_data.assign(temperature=reg_data.temperature_forecast)).loc[gbm_forecast.index]
rows["lesson 2 regression (for comparison)"] = {"MAE_2025": np.abs(reg_forecast - y[test]).mean(), "train_s": np.nan}
pd.DataFrame(rows).T.round(1)

                                      MAE_2025  train_s
calendar                                 121.6      1.7
+ temperature forecast                    81.0      2.2
+ demand lags (all features)              76.7      2.7
lesson 2 regression (for comparison)      73.6      NaN

Boosting follows the same path as lesson 2's regression: the calendar alone gives 121.6 MW, the temperature forecast brings it to 81.0 MW, and the honest demand lags to **76.7 MW**. But the regression — 89 parameters, a spline for temperature — is **better, at 73.6 MW**. Two reasons. These data follow a smooth structure that a regression with the right shape captures closely, while trees approximate smooth curves with steps and need many hours per step. And the lags add only 4 MW: they carry the recent level of demand, but at a lead time of 13 to 36 hours little of it survives (lesson 2's φᵏ). Gradient boosting tends to earn its keep on real data with many interacting inputs — several weather stations, wind, cloud cover, prices, special events — where nobody can write down the right regression. Machine learning is a tool, not a guarantee.

### Leakage through features

Now add two features a data scientist new to forecasting might find natural: demand **one hour earlier** and **24 hours earlier**. In a table they look like any other column. For a day-ahead forecast they are values from the future — `available` says so:

In [4]:
leaky = features.assign(one_hour_ago=demand.shift(1), same_hour_yesterday=demand.shift(24)).dropna()
yl = demand.loc[leaky.index]
tr_l, te_l = leaky.index < "2025", leaky.index >= "2025"
leaky_model = HistGradientBoostingRegressor(max_iter=1500, learning_rate=0.02, random_state=0).fit(leaky[tr_l], yl[tr_l])
leaky_mae = np.abs(leaky_model.predict(leaky[te_l]) - yl[te_l]).mean()
imp = permutation_importance(leaky_model, leaky[te_l].iloc[::10], yl[te_l].iloc[::10], n_repeats=3, random_state=0,
                             scoring="neg_mean_absolute_error")
importance = pd.Series(imp.importances_mean, index=leaky.columns).sort_values(ascending=False)
print(f"with the two leaky features: MAE {leaky_mae:.1f} MW  (honest model: {rows['+ demand lags (all features)']['MAE_2025']:.1f} MW)")
print("permutation importance (MW of MAE lost when the feature is scrambled):")
print(importance.round(1).to_string())

with the two leaky features: MAE 27.7 MW  (honest model: 76.7 MW)
permutation importance (MW of MAE lost when the feature is scrambled):
one_hour_ago            267.5
hour                     46.4
temperature_forecast     11.8
daytype                   8.2
same_hour_yesterday       0.3
last_known                0.3
same_hour_last_week       0.2
same_hour_2_days_ago      0.1
day_of_year               0.1


With the two leaky features the model scores **27.7 MW** — apparently 2.8 times better than anything honest, and entirely fake. Permutation importance shows how it got there: scrambling the 1-hour lag costs 267.5 MW of accuracy, more than all other features together. The model has learned to copy the value an hour earlier, which at forecast time lies 13 to 36 hours in the future. The 24-hour lag scores almost nothing only because the 1-hour lag already contains its information.

Note what importance does and does not say: it measures how much the model **uses** a feature, never whether you will **have** it when the forecast is made. A strong leak often becomes the most important feature of all. The only defence is to check each feature's availability before training — the `available` test above — and to be suspicious of any result that looks too good.

### Validating in time

Before testing on 2025, a modeller usually picks settings by cross-validation on the training years. Compare the default of most machine-learning tutorials — **shuffled 5-fold** cross-validation — with scikit-learn's **TimeSeriesSplit**, which always validates on data later than the training part, and with the honest truth: the error on 2025.

In [5]:
columns = list(features.columns)
Xtr, ytr = data.loc[train, columns], y[train]
estimates = {}
for name, cv in {"shuffled 5-fold": KFold(5, shuffle=True, random_state=0), "TimeSeriesSplit (5 folds)": TimeSeriesSplit(5)}.items():
    fold_mae = []
    for a, b in cv.split(Xtr):
        model = HistGradientBoostingRegressor(max_iter=1500, learning_rate=0.02, random_state=0).fit(Xtr.iloc[a], ytr.iloc[a])
        fold_mae.append(np.abs(model.predict(Xtr.iloc[b]) - ytr.iloc[b]).mean())
    estimates[name] = {"estimated MAE": np.mean(fold_mae), "smallest training fold (days)": min(len(a) for a, _ in cv.split(Xtr)) / 24}
estimates["actual error on 2025"] = {"estimated MAE": rows["+ demand lags (all features)"]["MAE_2025"],
                                     "smallest training fold (days)": train.sum() / 24}
pd.DataFrame(estimates).T.round(1)

                           estimated MAE  smallest training fold (days)
shuffled 5-fold                     46.7                          579.2
TimeSeriesSplit (5 folds)           93.2                          120.7
actual error on 2025                76.7                          724.0

Shuffled cross-validation estimates **46.7 MW** for a model that achieves **76.7 MW** on 2025 — 39 % too optimistic. Shuffling puts neighbouring hours into training and validation; since an error persists for hours (lesson 2), each validation hour has almost-identical twins in the training data, and the model is tested on what it has effectively seen. TimeSeriesSplit errs the other way, **93.2 MW**: its first fold trains on only 121 days — not even one full year, so the model has never seen the season it is asked about. The honest estimate comes from validating on a **later, complete period** — for example training on 2023 and validating on 2024, as lesson 1's exercise did — or from TimeSeriesSplit with a minimum training length of a full year. Whatever the scheme, the rule is lesson 1's: validation data must lie in the future of the training data.

### Who wins — and an ensemble

In [6]:
ensemble = 0.5 * reg_forecast + 0.5 * gbm_forecast        # weight fixed in advance, not tuned on 2025
actual = y[test]
compare = pd.DataFrame({
    "regression (lesson 2)": {"MAE": np.abs(reg_forecast - actual).mean()},
    "gradient boosting": {"MAE": np.abs(gbm_forecast - actual).mean()},
    "50/50 ensemble": {"MAE": np.abs(ensemble - actual).mean()}}).T
corr = np.corrcoef(reg_forecast - actual, gbm_forecast - actual)[0, 1]
print(f"correlation of the two models' errors: {corr:.2f}")
compare.round(1)

correlation of the two models' errors: 0.57


                        MAE
regression (lesson 2)  73.6
gradient boosting      76.7
50/50 ensemble         68.8

In [7]:
fig, ax = plt.subplots(figsize=(9, 3.4))
week = slice("2025-02-03", "2025-02-09 23:00")
ax.plot(actual.loc[week], color="black", lw=1.4, label="actual")
ax.plot(reg_forecast.loc[week], lw=1, label="regression")
ax.plot(gbm_forecast.loc[week], lw=1, label="gradient boosting")
ax.set(ylabel="MW", title="One winter week of 2025: two models, two kinds of error")
ax.legend(fontsize=8, ncol=3)
plt.tight_layout()
plt.show()

Averaging the two forecasts gives **68.8 MW** — better than the regression (73.6 MW) and the boosting model (76.7 MW). The reason is in the correlation of their errors, 0.57: high enough that both miss the same big swings, low enough that a good share of each model's error is its own. The winter week shows both: on Monday 3 and Wednesday 5 February both under-forecast the peak — demand's own persistent noise, which no model can foresee — while elsewhere they disagree, for example on Saturday 8 February, when the boosting model overshoots the middle of the day. Averaging cancels the disagreements and keeps the shared misses. The weight of one half was fixed in advance; choosing it from data must again happen on a validation period, never on the test year (lesson 1's exercise).

## Check your understanding

**Features and leaks**

1. The 24-hour lag is available for the morning hours of a day-ahead forecast but not for the afternoon. Use `available` to explain why, and say what lag the afternoon hours may use instead.
2. Permutation importance ranked the leaky feature first. Why is a high importance **not** evidence that a feature is legitimate?

**Models and validation**

1. Shuffled cross-validation estimated a much lower error than the model achieved on 2025. Name two reasons, one about time and one about the data.
2. Gradient boosting lost to a regression with fewer parameters. When would you expect it to win?